In [1]:
import pandas as pd
import os

def excel_to_parquet_smart():
    input_path = "../data/manual/LIGIE_HTS_Dashboard_v6.xlsx"
    output_dir = "../data/parquet_v4/"

    os.makedirs(output_dir, exist_ok=True)

    print(f"Iniciando procesamiento de: {input_path}")
    
    try:
        xls = pd.ExcelFile(input_path)

        for sheet_name in xls.sheet_names:
            print(f"\nProcesando hoja: '{sheet_name}'...")
            
            # Limpiamos el nombre para el archivo final
            clean_name = sheet_name.lower().replace(" ", "_").replace("/", "-")
            output_path = os.path.join(output_dir, f"{clean_name}.parquet")
            
            # --- CONDICIÓN: REGLAS ESPECÍFICAS POR HOJA ---
            if sheet_name.lower() in ['ligie', 'hts']:
                print(" -> Regla especial detectada: Forzando todo a texto para preservar ceros.")
                
                # Leer estricto como string
                df = pd.read_excel(xls, sheet_name=sheet_name, dtype=str)
                
                # Limpiar la palabra "nan" que se genera en columnas de texto vacías
                df = df.replace(['nan', 'None', 'NaN'], '')
                df = df.fillna('')
                
            else:
                print(" -> Regla estándar: Preservar números, estandarizar tipos mixtos.")
                
                # Leer normal (Pandas infiere int/float/object)
                df = pd.read_excel(xls, sheet_name=sheet_name)
                
                # Convertir SOLO las columnas mixtas ('object') a string
                # Las columnas 100% numéricas se quedan como números reales
                object_cols = df.select_dtypes(include=['object']).columns
                for col in object_cols:
                    df[col] = df[col].astype(str)
            # ----------------------------------------------
            
            # Exportar a Parquet
            df.to_parquet(output_path, index=False, engine='pyarrow')
            
            print(f" -> ✅ Guardado exitosamente en: {output_path}")
            
        print("\n¡Proceso completado con éxito! Se aplicaron las reglas según la hoja.")
        
    except Exception as e:
        print(f"❌ Ocurrió un error inesperado: {e}")

if __name__ == "__main__":
    excel_to_parquet_smart()

Iniciando procesamiento de: ../data/manual/LIGIE_HTS_Dashboard_v6.xlsx

Procesando hoja: 'ligie'...
 -> Regla especial detectada: Forzando todo a texto para preservar ceros.
 -> ✅ Guardado exitosamente en: ../data/parquet_v4/ligie.parquet

Procesando hoja: 'hts'...
 -> Regla especial detectada: Forzando todo a texto para preservar ceros.
 -> ✅ Guardado exitosamente en: ../data/parquet_v4/hts.parquet

Procesando hoja: 'sec_122'...
 -> Regla estándar: Preservar números, estandarizar tipos mixtos.
 -> ✅ Guardado exitosamente en: ../data/parquet_v4/sec_122.parquet

Procesando hoja: 'metals'...
 -> Regla estándar: Preservar números, estandarizar tipos mixtos.
 -> ✅ Guardado exitosamente en: ../data/parquet_v4/metals.parquet

Procesando hoja: 'auto'...
 -> Regla estándar: Preservar números, estandarizar tipos mixtos.
 -> ✅ Guardado exitosamente en: ../data/parquet_v4/auto.parquet

Procesando hoja: 'mhdv'...
 -> Regla estándar: Preservar números, estandarizar tipos mixtos.
 -> ✅ Guardado exit